# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/720-hz/flyrank-ml-internship/blob/main/work/notebooks/Week%203/w03_feature_leakage_check.ipynb?flush_cache=true)

This is the feature vector and leakage/privacy check for the Growth/Recovery/Momentum Prediction lane
(the same `is_declining_label` target used throughout this repo). The leakage hunt here is the one the
capstone paper (Week 8) cites directly — the safe feature set and the two checks below are exactly what
`work/notebooks/Week 8/capstone.ipynb` and `reproduce.py` build on.

## 1. Build the feature vector

*Engineered features, categorical handling, fills.*

Target: `is_declining_label = (trend_direction == "down")`. Feature set below is **safe by construction** —
it excludes every trailing-90-day engagement column (`impressions_90d`, `clicks_90d`, `ctr`, `engagement_rate`,
`scroll_rate`, `ai_traffic_pct`, ...) because Section 3 shows those structurally overlap the label's own
defining window. What's left: position, age, freshness, size, and content economics — plus categorical
metadata and explicit missingness flags (never silent zero-fills).

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

import pandas as pd
import numpy as np

RANDOM_STATE = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"Rows: {len(df):,} | clients: {df['client_id'].nunique()} | base rate: {df['is_declining_label'].mean():.3f}")

Working dir: /home/claude/flyrank-ml-internship


Rows: 30,000 | clients: 32 | base rate: 0.542


In [2]:
# Safe feature set -- the one the capstone paper and reproduce.py both use.
SAFE_NUMERIC = ["avg_position", "content_age_days", "days_since_last_update",
                "word_count", "char_count", "search_volume", "competition", "cpc"]
CATEGORICAL = ["competition_level", "content_type", "main_intent", "provider_used", "model_used"]
MISSING_FLAG_COLUMNS = ["word_count", "char_count", "search_volume", "competition", "cpc",
                         "main_intent", "competition_level", "provider_used", "model_used"]

def engineer(frame, numeric_cols=SAFE_NUMERIC):
    out = frame.copy()
    for col in MISSING_FLAG_COLUMNS:
        out[f"has_{col}"] = out[col].notna().astype(int)          # explicit missingness flag, never a silent 0
    num = out[numeric_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
    flags = out[[f"has_{c}" for c in MISSING_FLAG_COLUMNS]]
    cat = pd.get_dummies(out[CATEGORICAL].astype(str), dummy_na=False, dtype=float)
    return pd.concat([num, flags, cat], axis=1)

feature_vector = engineer(df)
print(f"Feature vector shape: {feature_vector.shape[0]:,} rows x {feature_vector.shape[1]} columns")
print("\nFirst 8 columns:", feature_vector.columns[:8].tolist())
feature_vector.head(3)

Feature vector shape: 30,000 rows x 34 columns

First 8 columns: ['avg_position', 'content_age_days', 'days_since_last_update', 'word_count', 'char_count', 'search_volume', 'competition', 'cpc']


,avg_position,content_age_days,days_since_last_update,word_count,char_count,search_volume,competition,cpc,has_word_count,has_char_count,...,main_intent_informational,main_intent_navigational,main_intent_transactional,provider_used_google,provider_used_openai,model_used_gemini-2.5-flash,model_used_gemini-3-flash-preview,model_used_gpt-4o-mini,model_used_gpt-5-mini,model_used_unknown
0,10.6,187,20,3221.0,20457.0,10.0,0.67,2.05,1,1,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,20.3,445,25,2481.0,15562.0,90.0,0.01,0.05,1,1,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,36.5,141,20,3515.0,23643.0,0.0,0.00,0.00,1,1,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing? | Available at decision time? |
|---|---|---|---|
| `avg_position` | Trailing-90d average SERP position (0 = no real position data, handled as its own signal, not a rank) | 0 encodes "no data," never NaN | Yes — reflects the 90 days *before* the snapshot |
| `content_age_days` | Days since the page was first published | Rare | Yes |
| `days_since_last_update` | Days since the page's content was last edited | Rare | Yes |
| `word_count` / `char_count` | Page length | ~26% missing (25.7% by direct count) — `has_word_count`/`has_char_count` flags carry that signal instead of a silent fill | Yes |
| `search_volume`, `competition`, `cpc` | Keyword-economics metadata for the page's primary target term | Sparse for some rows — missingness flagged | Yes, keyword-level metadata, not traffic-derived |
| `content_type`, `main_intent`, `provider_used`, `model_used` | Categorical content metadata | Occasional NaN, one-hot encoded as its own category | Yes |

**Deliberately excluded from this vector (see Section 4):** every trailing-90-day engagement column
(`impressions_90d`, `clicks_90d`, `sessions_90d`, `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`,
...) and every FlyRank product-decision field (`health_score`, `priority_score`, `action_type`, refresh
flags — never shipped in this CSV, so nothing to strip out there). `content_id` / `client_id` are used only
for grouping and joins, never as model inputs.

## 3. The leakage hunt

*Attack my own features: label-derived columns, future windows, product flags. Show the test, not the assertion.*

**Timeline check.** The label comes from `trend_direction`, computed from
`trend_pct = (impressions_last_30d - impressions_prev_30d) / impressions_prev_30d` — a comparison of the
most recent 60 days. A 90-day trailing engagement column entirely *contains* that 60-day window. Checked
directly below: `impressions_90d` correlates at **r ≈ 0.98** with `impressions_last_30d + impressions_prev_30d`
(the exact columns the label is built from) — a structural window overlap, not a copied column. That is
exactly why the safe feature set in Section 1 excludes the whole trailing-90-day engagement family.

**Harness sanity check.** To confirm the audit would actually catch leakage if it were present, I smuggle
`trend_pct` itself in as the *only* feature and refit: this should come back at (or essentially at) AUC = 1.000,
the "suspiciously perfect" red flag — proof the check below has teeth, which is what makes the more moderate,
disclosed result in Section 4 trustworthy rather than a blind spot the harness would have missed.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# --- Window-overlap check: confirm it, don't assert it ---
df["last_plus_prev"] = df["impressions_last_30d"] + df["impressions_prev_30d"]
overlap_share = (df["impressions_90d"] >= df["last_plus_prev"] - 1).mean()
overlap_corr = df["impressions_90d"].corr(df["last_plus_prev"])
print(f"Rows where impressions_90d >= (impressions_last_30d + impressions_prev_30d): {overlap_share:.1%}")
print(f"Correlation, impressions_90d vs. the label's own defining window: r={overlap_corr:.3f}")

# --- Client-grouped split, same convention as every other week in this repo ---
clients = df["client_id"].drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(clients)
n_test_clients = max(1, round(len(shuffled) * 0.2))
test_clients = set(shuffled[:n_test_clients])
mask = df["client_id"].isin(test_clients)
train_df, test_df = df[~mask].reset_index(drop=True), df[mask].reset_index(drop=True)
y_train, y_test = train_df["is_declining_label"], test_df["is_declining_label"]

# --- Harness sanity check: deliberately smuggle the forbidden column in as the ONLY feature ---
X_leak_train = train_df[["trend_pct"]].fillna(0).to_numpy()
X_leak_test = test_df[["trend_pct"]].fillna(0).to_numpy()
leak_model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_STATE, n_jobs=-1)
leak_model.fit(X_leak_train, y_train)
leak_scores = leak_model.predict_proba(X_leak_test)[:, 1]
leak_auc = roc_auc_score(y_test, leak_scores)
print(f"\nSanity check -- trend_pct smuggled in as the ONLY feature: ROC-AUC = {leak_auc:.4f}"
      f" (expected ~1.0 -- confirms the harness catches real leakage when it's actually there)")

Rows where impressions_90d >= (impressions_last_30d + impressions_prev_30d): 100.0%
Correlation, impressions_90d vs. the label's own defining window: r=0.980



Sanity check -- trend_pct smuggled in as the ONLY feature: ROC-AUC = 1.0000 (expected ~1.0 -- confirms the harness catches real leakage when it's actually there)


## 4. What I excluded and why

- **Every trailing-90-day engagement column** (`impressions_90d`, `clicks_90d`, `pageviews_90d`,
  `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `ctr`,
  `engagement_rate`, `scroll_rate`, `ai_traffic_pct`) — Section 3 shows `impressions_90d` alone correlates
  at r ≈ 0.98 with the label's own defining 60-day window. Not a copied column, but a structural overlap
  real enough to disclose and exclude.
- **`trend_direction` and `trend_pct`** — the label's own source columns. Used only to build
  `is_declining_label`, never as a feature (Section 3's smuggling test shows exactly why: AUC → ~1.0
  the moment either one leaks in).
- **FlyRank product-decision fields** (`health_score`, `priority_score`, `action_type`, refresh-tier flags)
  — never shipped in this CSV in the first place, so there is nothing to strip out; noted here so the
  exclusion is explicit rather than accidental.
- **`content_id` / `client_id`** — opaque pseudonymous hashes, used only for grouping and the train/test
  split, never as model inputs.
- **Raw client names, domains, URLs, titles, or search queries** — never shipped anywhere in this repo's
  public CSV; confirmed absent from every committed output under `work/` and `docs/`.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.